## Module 0 — Setup

In [ ]:
import io, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("grayscale")
plt.rcParams["figure.figsize"] = (10, 4)
pd.set_option("display.max_columns", 20)

RATING_MIN, RATING_MAX = 0, 10
print("Setup complete ✔")

## Module 1 — Data module (load & inspect)

In [ ]:
def load_data(path=None):
    """Load the dataset.
    - path given        -> read that file (.xlsx/.xls/.csv)
    - path None + Colab -> open the upload dialog
    - path None + local -> look for movies.xlsx / movies.csv in the working folder
    """
    if path is None:
        try:
            from google.colab import files          # only exists in Colab
            uploaded = files.upload()
            path = next(iter(uploaded))
        except ImportError:
            path = next((p for p in ("movies.xlsx", "movies.csv") if os.path.exists(p)), None)
            if path is None:
                raise FileNotFoundError("Put movies.xlsx or movies.csv next to the notebook.")
    if path.lower().endswith((".xlsx", ".xls")):
        df = pd.read_excel(path)
    else:
        df = pd.read_csv(path)
    print(f"Loaded '{path}' -> {df.shape[0]} rows x {df.shape[1]} columns")
    return df


def inspect_data(df):
    """Structure, data types, missing values, duplicates and summary statistics."""
    print("Shape            :", df.shape)
    print("Columns          :", list(df.columns))
    print("\nData types:\n", df.dtypes)
    print("\nMissing values:\n", df.isnull().sum()[df.isnull().sum() > 0])
    print("\nDuplicate titles :", int(df.duplicated(subset="title").sum()))
    bad = int(((df["rating"] < RATING_MIN) | (df["rating"] > RATING_MAX)).sum())
    print("Invalid ratings  :", bad, f"(outside {RATING_MIN}-{RATING_MAX})")
    print("\nNumeric summary:\n", df.describe().round(2))
    return df.head()

## Module 2 — Cleaning module

In [ ]:
def cleaning_report(raw):
    """Count the problems in the raw data (before cleaning)."""
    return {
        "rows_before": len(raw),
        "duplicate_titles": int(raw.duplicated(subset="title").sum()),
        "missing_ratings": int(raw["rating"].isnull().sum()),
        "invalid_ratings": int(((raw["rating"] < RATING_MIN) | (raw["rating"] > RATING_MAX)).sum()),
    }


def clean(raw):
    """Step 1 drop repeated titles -> Step 2 fill missing ratings with median
       -> Step 3 remove ratings outside 0-10."""
    df = raw.drop_duplicates(subset="title").reset_index(drop=True)
    median_rating = df["rating"].median()
    df["rating"] = df["rating"].fillna(median_rating)
    df = df[(df["rating"] >= RATING_MIN) & (df["rating"] <= RATING_MAX)].reset_index(drop=True)
    df.attrs["median_rating"] = median_rating
    return df


def validate(df):
    """Post-cleaning checks (Section 3.8 of the documentation)."""
    checks = {
        "No missing ratings": df["rating"].isnull().sum() == 0,
        "No duplicate titles": df.duplicated(subset="title").sum() == 0,
        "Year is integer": str(df["year"].dtype).startswith("int"),
        "Ratings within 0-10": df["rating"].between(RATING_MIN, RATING_MAX).all(),
    }
    for name, ok in checks.items():
        print(("✔" if ok else "✘"), name)
    print("Rows:", len(df), "| rating range:", df["rating"].min(), "-", df["rating"].max())
    return all(checks.values())


def save_clean(df, path="movies_clean.csv"):
    df.to_csv(path, index=False)
    print("Saved ->", path)

## Module 3 — Analysis module (EDA)

In [ ]:
def average_rating(df):
    return round(df["rating"].mean(), 2)

def genre_breakdown(df):
    return (df.groupby("genre")["rating"].mean().round(2)
              .sort_values(ascending=False).rename("avg_rating").reset_index())

def votes_rating_correlation(df):
    return round(df["votes"].corr(df["rating"]), 2)

def movies_per_year(df):
    return df.groupby("year").size().rename("movie_count").reset_index()

def rating_per_year(df):
    return df.groupby("year")["rating"].mean().round(2).rename("avg_rating").reset_index()

def high_rated_count(df, threshold=7.0):
    return int((df["rating"] >= threshold).sum())

def country_breakdown(df):
    return (df.groupby("country")["rating"].mean().round(2)
              .sort_values(ascending=False).rename("avg_rating").reset_index())

def language_breakdown(df):
    return (df.groupby("language")["rating"].agg(["count", "mean"]).round(2)
              .sort_values("mean", ascending=False).reset_index())

def director_leaderboard(df, min_movies=5, top_n=10):
    g = df.groupby("director")["rating"].agg(["count", "mean"]).round(2)
    return g[g["count"] >= min_movies].sort_values("mean", ascending=False).head(top_n).reset_index()

def profit_analysis(df):
    """Extra: revenue - budget (million USD) by genre."""
    t = df.assign(profit_musd=df["revenue_musd"] - df["budget_musd"])
    return t.groupby("genre")["profit_musd"].mean().round(1).sort_values(ascending=False).reset_index()

def summary(df):
    g = genre_breakdown(df)
    return {
        "Clean records analysed": len(df),
        "Average rating": average_rating(df),
        "Median rating": round(df["rating"].median(), 2),
        "Best genre": f"{g.iloc[0]['genre']} ({g.iloc[0]['avg_rating']})",
        "Worst genre": f"{g.iloc[-1]['genre']} ({g.iloc[-1]['avg_rating']})",
        "Votes-rating correlation": votes_rating_correlation(df),
        "Movies rated 7 or above": f"{high_rated_count(df)} ({high_rated_count(df)/len(df):.1%})",
        "Genres compared": df["genre"].nunique(),
    }

## Module 4 — Query module (search, filter, sort, compare, export)

In [ ]:
def search_title(df, text):
    """Case-insensitive title search."""
    return df[df["title"].str.contains(text, case=False, na=False)]

def filter_movies(df, genre=None, year_from=None, year_to=None, min_rating=None,
                  max_rating=None, min_votes=None, country=None, language=None, certificate=None):
    """Combine any filters; leave a parameter as None to ignore it."""
    m = pd.Series(True, index=df.index)
    if genre:        m &= df["genre"].eq(genre)
    if country:      m &= df["country"].eq(country)
    if language:     m &= df["language"].eq(language)
    if certificate:  m &= df["certificate"].eq(certificate)
    if year_from:    m &= df["year"] >= year_from
    if year_to:      m &= df["year"] <= year_to
    if min_rating is not None: m &= df["rating"] >= min_rating
    if max_rating is not None: m &= df["rating"] <= max_rating
    if min_votes is not None:  m &= df["votes"] >= min_votes
    return df[m]

def top_movies(df, n=10, by="rating"):
    return df.sort_values(by, ascending=False).head(n)[["title", "genre", "year", "rating", "votes"]]

def bottom_movies(df, n=10):
    return df.sort_values("rating").head(n)[["title", "genre", "year", "rating", "votes"]]

def compare_genres(df, genre_a, genre_b):
    cols = ["rating", "votes", "duration_min", "budget_musd", "revenue_musd"]
    return df[df["genre"].isin([genre_a, genre_b])].groupby("genre")[cols].mean().round(2).T

def add_movie(df, **row):
    """Add a record, e.g. add_movie(df, title='Movie 2000', genre='Drama', year=2025, rating=8.1, votes=5000)."""
    if row.get("title") in set(df["title"]):
        print("Title already exists - not added."); return df
    if not RATING_MIN <= row.get("rating", -1) <= RATING_MAX:
        print("Rating must be between 0 and 10 - not added."); return df
    return pd.concat([df, pd.DataFrame([row])], ignore_index=True)

def update_rating(df, title, new_rating):
    if not RATING_MIN <= new_rating <= RATING_MAX:
        print("Rating must be between 0 and 10."); return df
    df = df.copy()
    hit = df["title"] == title
    if not hit.any(): print("Title not found."); return df
    df.loc[hit, "rating"] = new_rating
    return df

def delete_movie(df, title):
    return df[df["title"] != title].reset_index(drop=True)

def export(df, path="result.csv"):
    """Save any result table and (in Colab) download it."""
    df.to_csv(path, index=False)
    try:
        from google.colab import files
        files.download(path)
    except ImportError:
        print("Saved ->", path)

## Module 5 — Recommendation module

In [ ]:
def recommend(df, genre, top_n=5, vote_percentile=0.75):
    """Best-rated movies of a genre among the well-voted ones (votes >= 75th percentile)."""
    data = df[df["genre"] == genre]
    if data.empty:
        raise ValueError(f"Unknown genre '{genre}'. Choose from: {sorted(df['genre'].unique())}")
    cutoff = data["votes"].quantile(vote_percentile)
    data = data[data["votes"] >= cutoff]
    return data.sort_values("rating", ascending=False).head(top_n)[
        ["title", "genre", "year", "rating", "votes"]].reset_index(drop=True)

def recommend_advanced(df, genre, top_n=5, year_from=None, country=None, language=None):
    """Same recommender, but narrowed by optional year / country / language."""
    sub = filter_movies(df, year_from=year_from, country=country, language=language)
    return recommend(sub, genre, top_n)

## Module 6 — Visualisation module

In [ ]:
def plot_movies_per_year(df):
    d = movies_per_year(df)
    plt.figure(figsize=(7, 3.4))
    plt.plot(d["year"], d["movie_count"], color="black", marker="o", linewidth=1.5)
    plt.title("Movies per Year"); plt.xlabel("Year"); plt.ylabel("Number of movies")
    plt.grid(True, linestyle="--", alpha=0.5); plt.show()

def plot_rating_per_year(df):
    d = df.groupby("year")["rating"].mean()
    plt.figure(figsize=(7, 3.4))
    plt.plot(d.index, d.values, color="black", linestyle="--", marker="s", markersize=4)
    plt.title("Average Rating per Release Year"); plt.xlabel("Year"); plt.ylabel("Average rating")
    plt.grid(True, linestyle="--", alpha=0.5); plt.show()

def plot_genre_bar(df):
    g = genre_breakdown(df)
    plt.figure(figsize=(7, 3.4))
    plt.bar(g["genre"], g["avg_rating"], color="gray", edgecolor="black")
    plt.title("Average Rating by Genre"); plt.xlabel("Genre"); plt.ylabel("Average rating"); plt.show()

def plot_top_genre(df, genre="Drama", n=5):
    top = recommend(df, genre, n)
    plt.figure(figsize=(7, 3.2))
    plt.barh(top["title"][::-1], top["rating"][::-1], color="lightgray", edgecolor="black", hatch="//")
    plt.title(f"Top {n} {genre} Movies (well-voted only)"); plt.xlabel("Rating"); plt.show()

def plot_genre_pie(df):
    share = df["genre"].value_counts()
    greys = ["#ffffff", "#d9d9d9", "#bdbdbd", "#969696", "#636363", "#252525"]
    plt.figure(figsize=(5.2, 3.9))
    plt.pie(share, labels=share.index, autopct="%1.1f%%",
            colors=greys[:len(share)], wedgeprops={"edgecolor": "black"})
    plt.title("Share of Movies by Genre"); plt.show()

def plot_genre_box(df):
    order = list(genre_breakdown(df)["genre"])
    plt.figure(figsize=(7, 3.4))
    plt.boxplot([df[df["genre"] == g]["rating"] for g in order], tick_labels=order,
                patch_artist=True, boxprops=dict(facecolor="lightgray"),
                medianprops=dict(color="black"))
    plt.title("Rating by Genre"); plt.xlabel("Genre"); plt.ylabel("Rating"); plt.show()

def plot_rating_hist(df):
    plt.figure(figsize=(7, 3.4))
    plt.hist(df["rating"], bins=20, color="lightgray", edgecolor="black")
    plt.title(f"Rating Distribution of {len(df):,} Movies")
    plt.xlabel("Rating (0-10)"); plt.ylabel("Number of movies")
    plt.grid(True, linestyle="--", alpha=0.5); plt.show()

def plot_votes_vs_rating(df):
    plt.figure(figsize=(7, 3.4))
    plt.scatter(df["votes"], df["rating"], s=10, alpha=0.45, color="black")
    plt.title(f"Votes vs Rating (correlation = {votes_rating_correlation(df)})")
    plt.xlabel("Votes"); plt.ylabel("Rating")
    plt.grid(True, linestyle="--", alpha=0.4); plt.show()

def plot_country_bar(df):
    c = country_breakdown(df)
    plt.figure(figsize=(7, 3.4))
    plt.bar(c["country"], c["avg_rating"], color="lightgray", edgecolor="black", hatch="..")
    plt.title("Average Rating by Country"); plt.ylabel("Average rating"); plt.xticks(rotation=20); plt.show()

def plot_budget_vs_revenue(df):
    plt.figure(figsize=(7, 3.4))
    plt.scatter(df["budget_musd"], df["revenue_musd"], s=10, alpha=0.4, color="black")
    plt.title("Budget vs Revenue"); plt.xlabel("Budget (M USD)"); plt.ylabel("Revenue (M USD)")
    plt.grid(True, linestyle="--", alpha=0.4); plt.show()

def plot_dashboard(df, genre="Drama"):
    per_year = movies_per_year(df); genres = genre_breakdown(df); top = recommend(df, genre)
    fig, axs = plt.subplots(2, 2, figsize=(9, 5.4))
    axs[0, 0].hist(df["rating"], bins=20, color="lightgray", edgecolor="black")
    axs[0, 0].set_title("Rating Distribution")
    axs[0, 1].plot(per_year["year"], per_year["movie_count"], color="black", marker="o", markersize=3)
    axs[0, 1].set_title("Movies per Year")
    axs[1, 0].bar(genres["genre"], genres["avg_rating"], color="gray", edgecolor="black")
    axs[1, 0].set_title("Average Rating by Genre"); axs[1, 0].tick_params(axis="x", rotation=30)
    axs[1, 1].barh(top["title"][::-1], top["rating"][::-1], color="lightgray", edgecolor="black", hatch="//")
    axs[1, 1].set_title(f"Top 5 {genre} Movies")
    fig.suptitle("Movie Rating Analysis System - Dashboard")
    plt.tight_layout(); plt.show()

def plot_all(df):
    for f in (plot_movies_per_year, plot_rating_per_year, plot_genre_bar, plot_genre_pie,
              plot_genre_box, plot_rating_hist, plot_votes_vs_rating, plot_country_bar,
              plot_budget_vs_revenue):
        f(df)

## Module 7 — Main module (run the pipeline)
The upload dialog opens — choose **movies.xlsx**.

In [ ]:
def main(path=None):
    raw = load_data(path)
    print("\n--- Inspection (raw) ---")
    display(inspect_data(raw))
    print("\n--- Cleaning ---")
    rep = cleaning_report(raw)
    df = clean(raw)
    print("Rows before cleaning :", rep["rows_before"])
    print("Duplicate titles     :", rep["duplicate_titles"], "-> removed")
    print("Missing ratings      :", rep["missing_ratings"], "-> filled with median", round(df.attrs["median_rating"], 1))
    print("Invalid ratings      :", rep["invalid_ratings"], "-> removed")
    print("Rows after cleaning  :", len(df))
    validate(df)
    save_clean(df)
    return raw, df

raw, df = main()          # in Colab: pick movies.xlsx in the upload dialog

## Using the modules

### A. Statistics

In [ ]:
for k, v in summary(df).items():
    print(f"{k:26}: {v}")
display(genre_breakdown(df))
display(country_breakdown(df))
display(rating_per_year(df).head())
display(director_leaderboard(df))
display(profit_analysis(df))

### B. Search / filter / sort

In [ ]:
display(search_title(df, "Movie 87"))
display(filter_movies(df, genre="Drama", year_from=2015, min_rating=8, min_votes=5000).head(10))
display(top_movies(df, 10))
display(bottom_movies(df, 5))
display(compare_genres(df, "Drama", "Romance"))

### C. Recommendations

In [ ]:
display(recommend(df, "Drama"))
display(recommend(df, "Action", top_n=10))
display(recommend_advanced(df, "Comedy", year_from=2010, country="India"))

### D. Charts

In [ ]:
plot_dashboard(df)       # four-panel dashboard
plot_all(df)             # every individual chart

### E. Edit the dataset (add / update / delete) and export

In [ ]:
df2 = add_movie(df, title="Movie 2000", genre="Drama", year=2025, rating=8.1, votes=12000,
                duration_min=120, language="English", certificate="UA", director="Director 1",
                lead_actor="Actor 1", country="USA", budget_musd=90.0, revenue_musd=300.0)
df2 = update_rating(df2, "Movie 2000", 8.5)
display(search_title(df2, "Movie 2000"))
df2 = delete_movie(df2, "Movie 2000")
# export(recommend(df, "Drama"), "drama_top5.csv")   # uncomment to download a result

### F. Optional — interactive genre picker

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

dd = widgets.Dropdown(options=sorted(df["genre"].unique()), description="Genre:")
n  = widgets.IntSlider(value=5, min=1, max=15, description="Top N:")
out = widgets.Output()

def show(_=None):
    with out:
        clear_output(); display(recommend(df, dd.value, n.value))

dd.observe(show, "value"); n.observe(show, "value")
display(dd, n, out); show()